# Model Performance Analysis

Post-hoc analysis of model behavior on multiple-choice tasks. **This notebook does not run inference** — it reads the JSONL files written by [`finetune/run_evaluations.py`](../finetune/run_evaluations.py) (and optionally [`finetune/question_difficulty_sweep.py`](../finetune/question_difficulty_sweep.py)) and turns those into accuracy / calibration / position-bias plots and tables.

## How to use

1. **Generate eval data** on Runpod / vast.ai:
   ```bash
   python finetune/run_evaluations.py \
       --base_model meta-llama/Meta-Llama-3.1-8B-Instruct \
       --lora_repo your-org/your-lora --merge \
       --dataset data/SimpleMC.jsonl \
       --evaluate_base_first --sigma 0.5
   ```
   That writes `finetuned_evals/<timestamp>_<model>_<lora>_<dataset>_n<N>_comparison.jsonl`.

2. **Pull the JSONL locally** and point `EVAL_PATH` below at it.

3. **Run the cells.** Plots and tables fall out per-section.

## What's in each section

| Section | What it answers |
| --- | --- |
| 1. Load eval results | Read JSONL, split base vs. finetuned rows, summarize counts. |
| 2. Accuracy & position bias | Overall accuracy by (model, dataset); per-position conditional accuracy; chi-square uniformity test on predicted positions. |
| 3. Calibration & metacognition | Reliability diagram (top_prob vs. is_correct), ECE, Brier, plus entropy ↔ stated_confidence correlation. |
| 4. Delegation game | **Inactive** — delegate-game scripts were dropped. Stub kept for when it's rebuilt. |
| 5. ECT analysis | Reads `outputs/ECT/*.json` from the ECT runner; reports ECE/AUROC/Brier vs. an MC-entropy ceiling. |

## Notes for migrating from `analyze_activations.ipynb`

Sections 1.8 / 1.9 / 1.10 used to live in `analyze_activations.ipynb` and read from `paired_data.json` (the introspection-experiment output). Those exact cells are preserved in [`legacy_code/analyze_activations_pre_split.ipynb`](legacy_code/analyze_activations_pre_split.ipynb). The new convention here is to read `run_evaluations.py` JSONLs instead. They expose fields the legacy `paired_data.json` didn't (e.g., `probs_ABCD`, `top_prob`, `prob_gap`, `is_correct`, `expected_confidence`), so code is a little simpler.

## Section 1 — Load eval results

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Make analysis_helpers and core importable when this notebook lives in analysis/.
REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'analysis' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / 'analysis') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'analysis'))

from analysis_helpers import (
    load_evaluation_jsonl, latest_eval_log,
    calibration_table, expected_calibration_error, brier_score,
)

# Point this at the JSONL written by run_evaluations.py. Default: latest in
# finetuned_evals/. Override manually for a specific file.
EVAL_PATH = latest_eval_log(REPO_ROOT / 'finetuned_evals')
print('EVAL_PATH =', EVAL_PATH)

In [ ]:
# Load the JSONL. If the file came from a base-vs-finetuned comparison run,
# samples have type 'instruct_eval_sample' or 'finetuned_eval_sample'. We
# load both halves separately for side-by-side plots.
if EVAL_PATH is None:
    raise FileNotFoundError(
        'No JSONL found in finetuned_evals/. Run finetune/run_evaluations.py first.'
    )

df_all, summaries = load_evaluation_jsonl(EVAL_PATH)
df_base, _ = load_evaluation_jsonl(EVAL_PATH, type_filter='instruct')
df_ft,   _ = load_evaluation_jsonl(EVAL_PATH, type_filter='finetuned')

print(f'Total sample rows: {len(df_all)}')
print(f'  base/instruct:   {len(df_base)}')
print(f'  finetuned:       {len(df_ft)}')
print(f'Summary blobs:     {list(summaries.keys())}')
df_all.head(3)

## Section 2 — Accuracy & position bias

Two checks:
1. **Overall accuracy** per model split. Sanity check that the eval ran and the finetune helped.
2. **Position bias.** A model that always picks position B will have high accuracy on questions where B is correct and chance accuracy elsewhere. Chi-square on predicted-position counts catches the same failure mode independently of correctness.

In [ ]:
def position_diagnostics(df, label):
    if df.empty:
        print(f'{label}: no rows'); return
    n = len(df)
    acc = df['is_correct'].mean()
    print(f'\n=== {label} (n={n}) ===')
    print(f'  overall accuracy: {acc:.2%}')

    pred_counts = df['model_answer_position'].value_counts().reindex([0, 1, 2, 3], fill_value=0)
    expected = n / 4.0
    chi2 = float(((pred_counts - expected) ** 2 / expected).sum())
    verdict = 'uniform' if chi2 < 7.81 else 'biased'
    print(f'  predicted-position counts (pos0–3): {pred_counts.tolist()}')
    print(f'  chi-square: {chi2:.2f}  (crit p=0.05: 7.81 → {verdict})')

    print('  accuracy conditioned on correct position:')
    for p in range(4):
        sub = df[df['correct_answer_position'] == p]
        if not sub.empty:
            print(f'    correct=pos{p}: {sub["is_correct"].mean():.2%}  (n={len(sub)})')

position_diagnostics(df_base, 'base / instruct')
position_diagnostics(df_ft,   'finetuned')

In [ ]:
# Side-by-side bar charts: predicted-position counts, base vs. finetuned.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, df, label in zip(axes, [df_base, df_ft], ['base', 'finetuned']):
    if df.empty:
        ax.set_title(f'{label}: no data'); continue
    counts = df['model_answer_position'].value_counts().reindex([0, 1, 2, 3], fill_value=0)
    ax.bar(['A', 'B', 'C', 'D'], counts.values)
    ax.axhline(len(df) / 4.0, color='gray', linestyle=':', label='uniform')
    ax.set_title(f'{label} predicted positions  (n={len(df)})')
    ax.set_ylabel('count')
    ax.legend()
fig.tight_layout(); plt.show()

## Section 3 — Calibration & metacognition

Reliability diagram on `top_prob` (the model's MC probability for its chosen answer) vs. `is_correct`. ECE summarizes the gap.

Stated-confidence calibration uses the model's `expected_confidence` field (computed from the 8-bin self-confidence pass in `evaluate_model`). The Spearman correlation between answer entropy and stated confidence is the standard metacognition signal — high (negative) correlation means stated confidence tracks internal uncertainty.

In [ ]:
def calibration_panel(df, label):
    if df.empty or 'top_prob' not in df.columns:
        print(f'{label}: missing top_prob'); return None
    valid = df.dropna(subset=['top_prob', 'is_correct'])
    tab = calibration_table(valid['top_prob'].to_numpy(),
                            valid['is_correct'].astype(float).to_numpy())
    ece = expected_calibration_error(valid['top_prob'].to_numpy(),
                                     valid['is_correct'].astype(float).to_numpy())
    brier = brier_score(valid['top_prob'].to_numpy(),
                        valid['is_correct'].astype(float).to_numpy())
    print(f'{label}: ECE={ece:.4f}  Brier={brier:.4f}  (n={len(valid)})')
    return tab

tab_base = calibration_panel(df_base, 'base')
tab_ft   = calibration_panel(df_ft,   'finetuned')

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color='gray', linestyle=':', label='perfect calibration')
for tab, label, color in [(tab_base, 'base', 'tab:blue'), (tab_ft, 'finetuned', 'tab:orange')]:
    if tab is None or tab.empty:
        continue
    ax.plot(tab['mean_conf'], tab['accuracy'], 'o-', label=label, color=color)
ax.set_xlabel('mean predicted probability (top_prob)')
ax.set_ylabel('empirical accuracy')
ax.set_title('Reliability diagram')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Metacognition: Spearman(entropy, stated_confidence). Negative = the model's
# stated confidence tracks its internal uncertainty (more entropy → less
# confidence). Requires the eval to have been run with --no_confidence absent.
for df, label in [(df_base, 'base'), (df_ft, 'finetuned')]:
    if df.empty or 'expected_confidence' not in df.columns or 'entropy' not in df.columns:
        print(f'{label}: skipping (no stated-confidence column)'); continue
    valid = df.dropna(subset=['entropy', 'expected_confidence'])
    if valid.empty:
        print(f'{label}: no valid rows'); continue
    rho, p = spearmanr(valid['entropy'], valid['expected_confidence'])
    print(f'{label}: Spearman(entropy, stated_conf) = {rho:+.3f}  (p={p:.2g}, n={len(valid)})')

## Section 4 — Delegation game (placeholder)

The delegate-game pipeline was removed in commit `8f1441d`. Once it's rebuilt for the new finetune, the analysis would go here — typical plot is delegate-rate vs. confidence bin. The legacy implementation lives in [`legacy_code/analyze_activations_pre_split.ipynb`](legacy_code/analyze_activations_pre_split.ipynb) under "Section 1.10".

## Section 5 — Explicit Confidence Task (ECT) analysis

Reads ECT result files (one JSON per run condition) from `outputs/ECT/`. Each row in a file has the model's MC probabilities, its stated confidence on the S–Z scale, the predicted answer letter, and `is_correct`. We compute:

- Stated-confidence calibration: ECE, Brier, AUROC against `is_correct`.
- **MC-entropy AUROC ceiling.** How well does the model's MC entropy alone predict correctness? Stated-confidence AUROC can't exceed this. Closing the gap = introspective access; raising the ceiling = better underlying calibration.
- Spearman / Pearson correlation between stated confidence and inverted MC entropy.

Originally lived in `analyze_ECT.ipynb` (now in `legacy_code/`).

In [ ]:
from analysis_helpers import load_ect_results
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score

ECT_DIR = REPO_ROOT / 'outputs' / 'ECT'
ect_files = sorted(ECT_DIR.glob('*.json')) if ECT_DIR.exists() else []
print(f'Found {len(ect_files)} ECT files in {ECT_DIR}')

if not ect_files:
    print('No ECT result files yet. Run the ECT pipeline first.')
else:
    ect = load_ect_results(ect_files)
    rows = []
    for name, blob in ect.items():
        data = blob.get('data', [])
        if not data:
            continue
        is_correct = np.array([int(d['is_correct']) for d in data])
        stated = np.array([float(d['stated_confidence_value']) for d in data])
        mc_entropy = np.array([
            -np.sum(np.array(d['mc_probs']) * np.log2(np.maximum(np.array(d['mc_probs']), 1e-12)))
            for d in data
        ])
        # Calibration of stated confidence (assumes stated is in [0, 1] already).
        ece = expected_calibration_error(stated, is_correct.astype(float))
        brier = brier_score(stated, is_correct.astype(float))
        auroc_stated = (roc_auc_score(is_correct, stated)
                        if len(np.unique(is_correct)) == 2 else float('nan'))
        # Ceiling: how well does internal entropy predict correctness?
        auroc_entropy = (roc_auc_score(is_correct, -mc_entropy)
                         if len(np.unique(is_correct)) == 2 else float('nan'))
        rho, _ = pearsonr(stated, -mc_entropy)
        rows.append({
            'run': name,
            'n': len(is_correct),
            'accuracy': is_correct.mean(),
            'mean_stated_conf': stated.mean(),
            'ece_stated': ece,
            'brier_stated': brier,
            'auroc_stated': auroc_stated,
            'auroc_entropy_ceiling': auroc_entropy,
            'pearson(stated, -entropy)': rho,
        })
    pd.DataFrame(rows).set_index('run')

---

**See also:** [`analyze_activations.ipynb`](analyze_activations.ipynb) for probe / direction analysis, [`analyze_interventions.ipynb`](analyze_interventions.ipynb) for ablation & steering analysis, [`legacy_code/`](legacy_code/) for the older implementations these notebooks superseded.